<h1 style="color:#c8a2c8; font-weight:bold; margin-bottom: 0px; padding-bottom: 0px;">ImmoEliza Project</h1>  
<h2 style="color:#c8a2c8; margin-top: 0px; padding-top: 0px;">ML content</h2>

### <span style="color:#c8a2c8; font-weight:bold">1 - Preprocessing</span>

In [2]:
# import libraries

# Standard 
import warnings 

# Data manipulation & visualization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
# other sklearn imports
from sklearn.base import BaseEstimator, TransformerMixin

# Display settings
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.3f}".format)
sns.set_theme(style="whitegrid")

# Reproducibility - alwas se a random seed
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("✅ Setup complete")

✅ Setup complete


In [3]:
# dataset loading

DATASET_CSV_PATH = "../data/raw/properties_final_irene.csv"
MANIP_DATASET_CSV_PATH = "../data/cleaned/properties_final_irene_after_analysis.csv"
properties = pd.read_csv(DATASET_CSV_PATH)
properties_original = properties.copy()

### <span style="color:#c8a2c8">1.1 - EDA</span>

In [4]:
print(f'Dataset Shape: {properties.shape[0]} rows x {properties.shape[1]} columns\n')
properties.head()


Dataset Shape: 12399 rows x 34 columns



,property_type,property_subtype,price,living_area_m2,bedrooms,bathrooms,address,postal_code,city,latitude,longitude,building_year,state_of_the_building,furnished,has_garage,parking_count,kitchen_equipped,has_elevator,facades,floors_total,has_garden,garden_area_m2,has_terrace,total_area_m2,epc_score,region,province,nearby_city,km_from_nearby_city,is_nearby_city_prestigious,floor_number,property_url,coord_swapped,price_per_m2
0,House,residence,550000.000,250.000,4.000,1.000,Rue des Nonnes 32A,4400,Flemalle,50.601,5.421,2019.000,New,0,1,1.000,Fully equipped,0,4.000,2.000,1,250.000,1,780.000,A+,Wallonia,Liège,Liège,11.800,0.000,NaN,https://immovlan.be/en/detail/residence/for-sa...,False,2200.000
1,Apartment,apartment,200000.000,57.000,1.000,1.000,Avenue Louise 32A,1050,Elsene,50.835,4.357,NaN,Normal,0,0,NaN,NaN,1,2.000,NaN,0,NaN,1,NaN,B,Brussels,Brussels Capital Region,Bruxelles,1.800,0.000,2.000,https://immovlan.be/en/detail/apartment/for-sa...,False,3509.000
2,Apartment,duplex,189000.000,115.000,1.000,1.000,NaN,7500,Tournai,50.606,3.388,2010.000,Excellent,0,0,NaN,NaN,0,NaN,NaN,0,NaN,1,NaN,G,Wallonia,Hainaut,Tournai,0.100,0.000,NaN,https://immovlan.be/en/detail/duplex/for-sale/...,False,1643.000
3,House,residence,549000.000,399.000,3.000,3.000,Kerkstraat 20 + 20/1,3941,Eksel,51.153,5.391,NaN,Normal,0,1,NaN,NaN,0,4.000,3.000,1,NaN,1,590.000,NaN,Flanders,Limburg,Genk,22.100,0.000,NaN,https://immovlan.be/en/detail/residence/for-sa...,False,1376.000
4,House,residence,330000.000,99.000,3.000,1.000,Quartier de Lohan 1,6980,La Roche En Ardenne,50.180,5.607,NaN,Excellent,1,1,1.000,Fully equipped,0,4.000,NaN,1,NaN,1,1036.000,D,Wallonia,Luxembourg,Verviers,48.900,0.000,NaN,https://immovlan.be/en/detail/residence/for-sa...,False,3333.000


In [5]:
properties.info()

<class 'pandas.DataFrame'>
RangeIndex: 12399 entries, 0 to 12398
Data columns (total 34 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   property_type               12399 non-null  str    
 1   property_subtype            12399 non-null  str    
 2   price                       12399 non-null  float64
 3   living_area_m2              11557 non-null  float64
 4   bedrooms                    11987 non-null  float64
 5   bathrooms                   11036 non-null  float64
 6   address                     9949 non-null   str    
 7   postal_code                 12399 non-null  int64  
 8   city                        12399 non-null  str    
 9   latitude                    12397 non-null  float64
 10  longitude                   12397 non-null  float64
 11  building_year               7433 non-null   float64
 12  state_of_the_building       9541 non-null   str    
 13  furnished                   12399 non-null

In [6]:
properties.describe(include="all")


,property_type,property_subtype,price,living_area_m2,bedrooms,bathrooms,address,postal_code,city,latitude,longitude,building_year,state_of_the_building,furnished,has_garage,parking_count,kitchen_equipped,has_elevator,facades,floors_total,has_garden,garden_area_m2,has_terrace,total_area_m2,epc_score,region,province,nearby_city,km_from_nearby_city,is_nearby_city_prestigious,floor_number,property_url,coord_swapped,price_per_m2
count,12399,12399,12399.000,11557.000,11987.000,11036.000,9949,12399.000,12399,12397.000,12397.000,7433.000,9541,12399.000,12399.000,4222.000,3375,12399.000,9375.000,5581.000,12399.000,2780.000,12399.000,6892.000,10445,12399,12399,12397,12397.000,12397.000,3508.000,12399,12399,11557.000
unique,2,15,NaN,NaN,NaN,NaN,9378,NaN,1601,NaN,NaN,NaN,8,NaN,NaN,NaN,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11,3,11,44,NaN,NaN,NaN,12390,2,NaN
top,House,residence,NaN,NaN,NaN,NaN,Dessauerplein 1,NaN,Liege,NaN,NaN,NaN,Normal,NaN,NaN,NaN,Fully equipped,NaN,NaN,NaN,NaN,NaN,NaN,NaN,C,Wallonia,Liège,Bruxelles,NaN,NaN,NaN,https://immovlan.be/en/detail/residence/for-sa...,False,NaN
freq,8023,6506,NaN,NaN,NaN,NaN,33,NaN,267,NaN,NaN,NaN,3665,NaN,NaN,NaN,1604,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2093,5988,1722,1386,NaN,NaN,NaN,2,12355,NaN
mean,NaN,NaN,399453.556,173.570,3.078,1.411,NaN,5014.645,NaN,50.720,4.549,1970.028,NaN,0.036,0.451,1.861,NaN,0.193,2.959,3.314,0.537,1232.853,0.658,1560.219,NaN,NaN,NaN,NaN,35.256,0.063,3.541,NaN,NaN,2579.398
std,NaN,NaN,370800.032,156.447,2.276,1.168,NaN,2678.774,NaN,0.372,0.817,47.241,NaN,0.186,0.498,29.254,NaN,0.395,0.880,2.936,0.499,5413.624,0.474,5068.652,NaN,NaN,NaN,NaN,388.588,0.243,29.263,NaN,NaN,1397.460
min,NaN,NaN,19900.000,12.000,1.000,1.000,NaN,1000.000,NaN,49.507,2.509,1500.000,NaN,0.000,0.000,1.000,NaN,0.000,1.000,1.000,0.000,1.000,0.000,1.000,NaN,NaN,NaN,NaN,0.000,0.000,0.000,NaN,NaN,106.000
25%,NaN,NaN,195000.000,90.000,2.000,1.000,NaN,2570.000,NaN,50.472,4.156,1950.000,NaN,0.000,0.000,1.000,NaN,0.000,2.000,2.000,0.000,117.000,0.000,267.750,NaN,NaN,NaN,NaN,3.200,0.000,1.000,NaN,NaN,1702.000
50%,NaN,NaN,302500.000,145.000,3.000,1.000,NaN,4960.000,NaN,50.772,4.462,1975.000,NaN,0.000,0.000,1.000,NaN,0.000,3.000,3.000,1.000,352.500,1.000,629.000,NaN,NaN,NaN,NaN,8.400,0.000,2.000,NaN,NaN,2400.000
75%,NaN,NaN,499000.000,207.000,4.000,2.000,NaN,7100.000,NaN,51.009,5.238,2007.000,NaN,0.000,1.000,1.000,NaN,0.000,4.000,4.000,1.000,900.000,1.000,1316.250,NaN,NaN,NaN,NaN,17.000,0.000,3.000,NaN,NaN,3158.000


In [9]:
columns = [
    'property_type', 
    'property_subtype', 
    'state_of_the_building', 
    'furnished', 
    'has_garage', 
    'kitchen_equipped', 
    'has_elevator', 
    'has_garden', 
    'has_terrace', 
    'epc_score', 
    'region', 
    'province', 
    'is_nearby_city_prestigious'
]
for column in columns:
    elements = "\n".join(properties[column].dropna().unique().astype(str))
    print(f'{column} in the dataset contains {properties[column].nunique()} unique values:\n{elements}')
    print(f'Datatype of vales in the column is {properties[column].dropna().unique().dtype}.')
    print("\n")

property_type in the dataset contains 2 unique values:
House
Apartment
Datatype of vales in the column is str.


property_subtype in the dataset contains 15 unique values:
residence
apartment
duplex
mixed-building
bungalow
villa
penthouse
cottage
chalet
studio
ground-floor
triplex
master-house
mansion
loft
Datatype of vales in the column is str.


state_of_the_building in the dataset contains 8 unique values:
New
Normal
Excellent
To renovate
Fully renovated
To restore
Under construction
To demolish
Datatype of vales in the column is str.


furnished in the dataset contains 2 unique values:
0
1
Datatype of vales in the column is int64.


has_garage in the dataset contains 2 unique values:
1
0
Datatype of vales in the column is int64.


kitchen_equipped in the dataset contains 4 unique values:
Fully equipped
Super equipped
Partially equipped
Not equipped
Datatype of vales in the column is str.


has_elevator in the dataset contains 2 unique values:
0
1
Datatype of vales in the column is 

In [ ]:
# properties[column] = properties[column].astype('Int64')

<h3 style="color:#c8a2c8">1.2 - Drop Useless Columns</h3>  

I'm going to remove the columns:  
- `'property_url'`;  
- `'address'`;  
- `'price_per_m2'`

In [10]:
# removal of columns that a model can't calculate directly:
# 'property_url', 'address'

# removal of 'price_per_m2' also to prevent Data Leakage

# create a dedicated config dictionary
COLS_TO_DROP = {
    "'price_per_m2'": "prevent Data Leakage",
    "'property_url'": "Unique for each property - no predictive power",
    "'address'": "Unique for each property - no predictive power"
}

# Drop only columns that exist (safe for re-runs)
cols_present = [c for c in COLS_TO_DROP if c in properties.columns]
properties = properties.drop(columns=cols_present)

print(f'Dropped {len(cols_present)} column(s): {", ".join(cols_present)}')
print(f'Remaining dataset Shape: {properties.shape}')

Dropped 0 column(s): 
Remaining dataset Shape: (12399, 34)


<h3 style="color:#c8a2c8">1.3 - Feature Engineering — Creating New Features</h3>  

I want to add:
-  a categorical column named `'urban_density'` reporting the level of urban density (between High, Medium, Low) according to corresponding `'postal_code'` of the property;  
- a column named `'property_age'` (2026 - df['building_year']);  
- a column named `'living_area_ratio'` (df['living_area_m2'] / df['total_area_m2']);  
- a column named `'bedroom_density'` (df['bedrooms'] / df['living_area_m2']).

In [11]:
'''
===========================
'urban_density' column
===========================
'''

# importing different pandas dataframe with demografical data of Belgium
# source: StatBel

# https://statbel.fgov.be/en/open-data/population-statistical-sector-12
DEMOG_BE_DATASET = "../data/raw/StatBel/population_data/OPENDATA_SECTOREN_2025_NEW.txt"
# https://statbel.fgov.be/en/open-data/address-file-statistical-sector
NIS_CODES_DATASET = "../data/raw/StatBel/population_data/TF_RGN_ADDRESSES_STAT_SECTORS_20240401.txt"

demog_df = pd.read_csv(DEMOG_BE_DATASET, sep='|', encoding='cp1252')
nis_df = pd.read_csv(NIS_CODES_DATASET, sep='|', encoding='cp1252')

demog_df.info()
print("--"*30)
nis_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 21183 entries, 0 to 21182
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   CD_REFNIS            21183 non-null  int64
 1   CD_SECTOR            21183 non-null  str  
 2   TOTAL                21183 non-null  int64
 3   DT_STRT_SECTOR       21183 non-null  str  
 4   DT_STOP_SECTOR       21183 non-null  str  
 5   OPPERVLAKKTE IN HM²  21183 non-null  str  
 6   TX_DESCR_SECTOR_NL   21183 non-null  str  
 7   TX_DESCR_SECTOR_FR   21183 non-null  str  
 8   TX_DESCR_NL          21183 non-null  str  
 9   TX_DESCR_FR          21183 non-null  str  
dtypes: int64(2), str(8)
memory usage: 1.6 MB
------------------------------------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 4451153 entries, 0 to 4451152
Data columns (total 21 columns):
 #   Column              Dtype  
---  ------              -----  
 0   CD_REFNIS           int64  
 1   TX_REFNIS_DES

demog_df adaptation

In [12]:
demog_df['OPPERVLAKKTE IN HM²'].head(5)

0    52,9876523968054
1    67,2440779981788
2    28,0836525518644
3    42,8105162953392
4    25,4857858896115
Name: OPPERVLAKKTE IN HM², dtype: str

In [13]:
# elements in demog_df['OPPERVLAKKTE IN HM²'] are strings and have a comma "," instead of dot "."
# I replace it and convert the datatype to float
demog_df['OPPERVLAKKTE IN HM²'] = demog_df['OPPERVLAKKTE IN HM²'].str.replace(',', '.').astype(float)
demog_df['OPPERVLAKKTE IN HM²'].head(5)

0   52.988
1   67.244
2   28.084
3   42.811
4   25.486
Name: OPPERVLAKKTE IN HM², dtype: float64

In [14]:
# add a new column to gave the info in KM² instead
demog_df['surface_KM²'] = demog_df['OPPERVLAKKTE IN HM²'] / 100
demog_df['surface_KM²'].head(5)

0   0.530
1   0.672
2   0.281
3   0.428
4   0.255
Name: surface_KM², dtype: float64

<span style="color:red">FAILED</span>: I try to merge the postal code column from nis_df to demo_df through CD_SECTOR

In [ ]:
# I need population info related to postal code

# forcing conversion to str removing hidden empty spaces
nis_df['CD_SECTOR_2023'] = nis_df['CD_SECTOR_2023'].astype(str).str.strip()
demog_df['CD_SECTOR'] = demog_df['CD_SECTOR'].astype(str).str.strip()
# extract an unique mapping of postal code (CD_ZIP) for each sector (CD_SECTOR_2023) 
bridge_df = nis_df[['CD_SECTOR_2023', 'CD_ZIP']].drop_duplicates()
# renominate the column from nis_df to make it identical to the one in demog_df
bridge_df = bridge_df.rename(columns={'CD_SECTOR_2023': 'CD_SECTOR'})

In [ ]:
# joining brifge_df to demog_df
demog_df_complete = demog_df.merge(bridge_df, on='CD_SECTOR', how='inner')
print(f'Number of rows in former demog_df: {demog_df.shape[0]}')
print(f'Number of rows in new demog_df_complete: {demog_df_complete.shape[0]}')

In [ ]:
# diagnostic cell - investigate discrepancy in CD_SCETOR

# printing a sample from both starting StatBel datasets
print(f'Sample of demog_df: {demog_df['CD_SECTOR'].dropna().head(5).tolist()}')
print(f'Sample of nis_df: {nis_df['CD_SECTOR_2023'].dropna().head(5).tolist()}')

# check the length of characters
print(f'String length in demog_df: {demog_df['CD_SECTOR'].str.len().unique()}')
print(f'String length in nis_df: {nis_df['CD_SECTOR_2023'].str.len().unique()}')

<span style="color:red">My attempt failed, CD_SECTOR from demog_df and CD_SECTOR_2023 from nis_df mismatch.</span>  
  
<span style="color:green">ADOPTED APPROACH</span>: Comune (CD_REFNIS) is the only geographical reference that the twoo datasets share: so I aggregate on comune level.

In [ ]:
# StatBel datasets are organized in micro-sectors
# so I need to calculate population amount and surface area in total for each comune
comune_df = demog_df.groupby('CD_REFNIS').agg(
    tot_pop_per_comune=('TOTAL', 'sum'),
    tot_area_km2=('surface_KM²', 'sum')
).reset_index()

In [16]:
# calculate and adding 'urban_density' column to comune_df
# values are 1 per comune (average representative value)

comune_df['urban_density'] = comune_df['tot_pop_per_comune'] / comune_df['tot_area_km2']

In [ ]:
# isolate from nis_df correspondance between CD_ZIP and CD_REFNIS
bridge_df = nis_df[['CD_ZIP', 'CD_REFNIS']].drop_duplicates()
# add the comune_df to bridge_df to implement 'urban_density' info
df_zip_density = bridge_df.merge(comune_df[['CD_REFNIS', 'urban_density']], on='CD_REFNIS', how='inner')

# in Belgium postal_code may intersect more than one comune
# so, in order to avoid having same CD_ZIP associated to different densities, 
# final aggregation on 'CD_ZIP' calculating mean value
density_mapping = df_zip_density.groupby('CD_ZIP')['urban_density'].mean()

In [ ]:
# mapping 'urban_density' to properties dataset
properties['urban_density'] = properties['postal_code'].map(density_mapping)

# checking new column state

properties['urban_density'].head(5)
print(f'Number on NaN values: {properties['urban_density'].isna().sum()}')
print(f'Proportion of NaN values: {(properties['urban_density'].isna().mean()*100).round(2)}%')
# check if association with 'postal_code' has geometrical sense
properties[['postal_code', 'urban_density']].drop_duplicates().head(10)
# extract postal codes list with NaN urban density
pc_nan_dens = properties[properties['urban_density'].isna()]['postal_code'].unique().astype(str)
print(f'Postal codes with nan as urban density:\n{"\n".join(pc_nan_dens)}')

Adding also `'property_age'`, `'living_area_ratio'` and `'bedroom_density'` new columns.

In [ ]:
'''
===========================
'property_age' column
===========================
'''
# 2026 - df['building_year']

properties['property_age'] = 2026 - properties['building_year']

'''
===========================
'living_area_ratio' column
===========================
'''
# df['living_area_m2'] / df['total_area_m2']

properties['living_area_ratio'] = properties['living_area_m2'] / properties['total_area_m2']

'''
===========================
'bedroom_density' column
===========================
'''
# df['bedrooms'] / df['living_area_m2']

properties['bedroom_density'] = properties['bedrooms'] / properties['living_area_m2']

In [ ]:
properties.describe(include='all')

<h3 style="color:#c8a2c8">1.4 - Encoding Categorical Features</h3>

Using <span style="color:#8064A2; font-weight:bold">Label / Ordinal Encoding</span> for variables that categorize data according to a specific sequence (**ordinal variables**: the numerical order reflects reality - like `'epc_score'`) and <span style="color:#2E75B5; font-weight:bold">One-Hot Encoding</span> for nominal variables that don't (**nominal variables** - like `'property_subtype'`).

Categorical columns in properties df are:  
- `'property_type'` &rarr; `House` (<span style="color:orange">change to 0</span>) and `Apartment` (<span style="color:orange">change to 1</span>); 
- `'property_subtype'` (<span style="color:#2E75B5; font-weight:bold">One-Hot Encoding</span>) &rarr; `residence`, `apartment`, `duplex`, `mixed-building`, `bungalow`, `villa`, `penthouse`, `cottage`, `chalet`, `studio`, `ground-floor`, `triplex`, `master-house`, `mansion` and `loft`;
- `'state_of_the building'` (<span style="color:#8064A2; font-weight:bold">Label / Ordinal Encoding</span>) &rarr; <span style="color:orange">make a MAPPING DICTIONARY (unify `Fully renovated` and `Excellent`)</span>: `New`, `Under construction`, `Fully renovated`, `Excellent`, `Normal`, `To renovate`, `To restore` and `To demolish`;
- `'furnished'` &rarr; <span style="color:green">DONE</span>: already binary, dtype int64;
- `'has_garage'` &rarr; <span style="color:green">DONE</span>: already binary, dtype int64;
- `'kitchen_equipped'` (<span style="color:#8064A2; font-weight:bold">Label / Ordinal Encoding</span>) &rarr; <span style="color:orange">make a MAPPING DICTIONARY</span>: `Super equipped`, `Fully equipped`, `Partially equipped` and `Not equipped`;
- `'has_elevator'` &rarr; <span style="color:green">DONE</span>: already binary, dtype int64;
- `'has_garden'` &rarr; <span style="color:green">DONE</span>: already binary, dtype int64;
- `'has_terrace'`  &rarr; <span style="color:green">DONE</span>: already binary, dtype int64;
- `'epc_score'` (<span style="color:#8064A2; font-weight:bold">Label / Ordinal Encoding</span>) &rarr; <span style="color:orange">add intermediate levels `A-`, `B-`, `C-`, `D-`, `E-`, `C+` and `D+` and make a MAPPING DICTIONARY</span>: `A++`, `A+`, `A`, `B+`, `B`, `C`, `D`, `E+`, `E`, `F`, `G`;
- `'region'` (<span style="color:#2E75B5; font-weight:bold">One-Hot Encoding</span>) &rarr; `Wallonia`, `Brussels` and `Flanders`;
- `'province'` (<span style="color:#2E75B5; font-weight:bold">One-Hot Encoding</span>) &rarr; `Liège`, `Brussels Capital`, `Region`, `Hainaut`, `Limburg`, `Luxembourg`, `Namur`, `East Flanders`, `Antwerp`, `West Flanders`, `Walloon Brabant` and `Flemish Brabant`;
- `'is_nearby_city_prestigious'` &rarr; <span style="color:yellow">CHANGE DTYPE TO INT64</span>: already binary, but change from float64 to int64;
- `'urban_density'` (<span style="color:#8064A2; font-weight:bold">Label / Ordinal Encoding</span>) &rarr; <span style="color:orange">CONVERT IT TO CATEGORIAL and make a MAPPING DICTIONARY</span>: continuous numerical variable, to be splitted in categories `LOW`, `MEDIUM` and `HIGH`.

<span style="color:#8064A2; font-weight:bold">Ordinal Encoding</span> is exclusively for the input features X (the feature columns, such as epc_score or state_of_the_building),